ELMo, BERT, and Sentence Transformers

In [1]:
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer
import tensorflow_hub as hub
import tensorflow as tf
from sklearn.metrics.pairwise import cosine_similarity

## Sample sentences for all examples

In [2]:
# Sample sentences for all examples
sentences = [
    "The cat sat on the mat.",
    "A cat rested on the rug.",
    "The dog ran in the park.",
    "It's raining today.",
    "The weather is nice."
]

# 1. ELMo - Embeddings from Language Models

Creates context-aware word embeddings using bidirectional LSTMs. The same word gets different representations based on surrounding context (e.g., "bank" means different things in "river bank" vs "money bank").


In [3]:
# Load ELMo from TensorFlow Hub
print("\n📚 Loading ELMo model...")
elmo = hub.load("https://tfhub.dev/google/elmo/3")
def get_elmo_embeddings(texts):
    """Get ELMo embeddings for text"""

    # Convert texts to the format ELMo expects
    if isinstance(texts, list):
        texts = tf.constant(texts)

    # Get embeddings using the correct method
    embeddings = elmo.signatures['default'](text=texts)['elmo']

    # Average across words to get sentence embeddings
    sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

    print(f"✅ ELMo embeddings shape: {sentence_embeddings.shape}")
    return sentence_embeddings.numpy()

# Example usage
try:
    elmo_embeddings = get_elmo_embeddings(sentences)
    print(f"Each sentence becomes a {elmo_embeddings.shape[1]}-dimensional vector")
except Exception as e:
    print(f"❌ ELMo failed: {e}")
    elmo_embeddings = None


📚 Loading ELMo model...
✅ ELMo embeddings shape: (5, 1024)
Each sentence becomes a 1024-dimensional vector


# 2. BERT - Bidirectional Encoder Representations from Transformers

Uses transformers to understand text by looking at words in both directions simultaneously. Pre-trained on massive text, then fine-tuned for specific NLP tasks like classification and question answering.


In [4]:
# Load BERT tokenizer and model
print("\n🤖 Loading BERT model...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
def get_bert_embeddings(texts):
    """Get BERT embeddings for text"""


    bert_model.eval()

    embeddings = []

    with torch.no_grad():
        for text in texts:
            # Tokenize text
            inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)

            # Get BERT outputs
            outputs = bert_model(**inputs)

            # Use [CLS] token embedding (first token)
            cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()
            embeddings.append(cls_embedding.flatten())

    embeddings = np.array(embeddings)
    print(f"✅ BERT embeddings shape: {embeddings.shape}")
    return embeddings

# Example usage
bert_embeddings = get_bert_embeddings(sentences)
print(f"Each sentence becomes a {bert_embeddings.shape[1]}-dimensional vector")


🤖 Loading BERT model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERT embeddings shape: (5, 768)
Each sentence becomes a 768-dimensional vector


# 3. Sentence Transformers - Easy and Powerful!

Modifies BERT to create meaningful sentence-level embeddings. Encodes entire sentences into vectors for tasks like semantic search and document similarity. Simple to use with strong performance.RetryClaude can make mistakes. Please double-check responses.

In [5]:
def get_sentence_transformer_embeddings(texts):
    """Get Sentence Transformer embeddings - the easiest way!"""
    print("\n⚡ Loading Sentence Transformer model...")

    # Load pre-trained model (this one is fast and good!)
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Get embeddings - super simple!
    embeddings = model.encode(texts)

    print(f"✅ Sentence Transformer embeddings shape: {embeddings.shape}")
    return embeddings

# Example usage
st_embeddings = get_sentence_transformer_embeddings(sentences)
print(f"Each sentence becomes a {st_embeddings.shape[1]}-dimensional vector")


⚡ Loading Sentence Transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Sentence Transformer embeddings shape: (5, 384)
Each sentence becomes a 384-dimensional vector


#

# Exercise

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Dataset

We use a sentiment review dataset and filter it to include only extreme sentiment categories:

- Very Bad Reviews (Rating: 1) - Negative class
- Very Good Reviews (Rating: 5) - Positive class

This creates a binary classification problem with clear sentiment distinction, allowing us to better evaluate the effectiveness of different text representation methods.

In [11]:
df = pd.read_csv("/content/drive/MyDrive/JCAIEAH-003/Notes and Hands On/Modul 2/Day 6/data/gojek_review.csv")
df = df[df.rate.isin([1, 5])].head(100) # Only classify very bad and very good rating
df.head()

,review,rate
0,Sangat kecewa. Kecewa sekali. Udh top up. Mau ...,1
1,Ga niat ngasih promo sialan temen udh pake ref...,1
2,Kalau sistemnya rata begini apa bedanya yg raj...,1
3,"Ongkosnya da mahal, minimal 16rb..... Sekarang...",1
4,Tolol anjing..!!!! Aplikasi yang katanga karya...,1


# Dataset Splitting

The filtered dataset is split into training and testing sets to ensure proper model evaluation:

- Training set: Used to train the classification models
- Test set: Used to evaluate model performance and compare different embedding methods

In [12]:
X = df.review
y = df.rate

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((80,), (20,), (80,), (20,))

# Evaluation Approach

All methods use the same classification model and parameters to ensure fair comparison. Performance metrics will help determine which text representation method works best for sentiment classification on this dataset.

# Method 1: Word2Vec

Dense vector representation that captures semantic relationships:

- Pre-trained or custom-trained word embeddings that represent words in continuous vector space
- Captures semantic similarity between words based on contextual usage
- Document representation typically created by averaging word vectors

In [13]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.3 MB/s eta 0:00:00


In [14]:
from gensim.models import Word2Vec
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
import nltk
nltk.download('stopwords')
sw_indo = stopwords.words("indonesian") + list(punctuation)
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [15]:
sentences = [word_tokenize(rev.lower()) for rev in X_train]
model = Word2Vec(sentences=sentences, vector_size=128, window=5, min_count=3, workers=4, epochs=1000)
w2v = model.wv

In [16]:
vecs = []
for sentence in X_train:
    word_vector = np.zeros(128)
    for word in sentence:
        if word not in sw_indo and word in w2v.index_to_key:
            word_vector += w2v[word]
    vecs.append(word_vector)
X_train_vector = np.array(vecs)

In [17]:
vecs = []
for sentence in X_test:
    word_vector = np.zeros(128)
    for word in sentence:
        if word not in sw_indo and word in w2v.index_to_key:
            word_vector += w2v[word]
    vecs.append(word_vector)
X_test_vector = np.array(vecs)

In [18]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 0.9
Test: 0.85



# Method 2: ELMo

In [19]:
X_train_vector = get_elmo_embeddings(X_train.values)

✅ ELMo embeddings shape: (80, 1024)


In [20]:
X_test_vector = get_elmo_embeddings(X_test.values)

✅ ELMo embeddings shape: (20, 1024)


In [21]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 1.0
Test: 0.85



# Method 3: BERT

In [22]:
batch_size = 10
X_train_vector = []
for i in range(0, len(X_train), batch_size):
    batch = X_train.iloc[i:i + batch_size]
    # Process the 'batch' DataFrame here
    print(f"Processing batch from index {i} to {i + len(batch) - 1}")
    # Example: print the first row of each batch
    # print(batch.iloc[0])
    X_train_vector.extend(get_bert_embeddings(batch.values))

Processing batch from index 0 to 9
✅ BERT embeddings shape: (10, 768)
Processing batch from index 10 to 19
✅ BERT embeddings shape: (10, 768)
Processing batch from index 20 to 29
✅ BERT embeddings shape: (10, 768)
Processing batch from index 30 to 39
✅ BERT embeddings shape: (10, 768)
Processing batch from index 40 to 49
✅ BERT embeddings shape: (10, 768)
Processing batch from index 50 to 59
✅ BERT embeddings shape: (10, 768)
Processing batch from index 60 to 69
✅ BERT embeddings shape: (10, 768)
Processing batch from index 70 to 79
✅ BERT embeddings shape: (10, 768)


In [23]:
X_test_vector = get_bert_embeddings(X_test.values)

✅ BERT embeddings shape: (20, 768)


In [24]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 1.0
Test: 0.85



# Method 4: Sentence Transformers

In [25]:
X_train_vector = get_sentence_transformer_embeddings(X_train.values)
X_test_vector = get_sentence_transformer_embeddings(X_test.values)


⚡ Loading Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Sentence Transformer embeddings shape: (80, 384)

⚡ Loading Sentence Transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Sentence Transformer embeddings shape: (20, 384)


In [26]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_vector, y_train)
print(f"""Model Performance (Accuracy):
Train: {model.score(X_train_vector, y_train)}
Test: {model.score(X_test_vector, y_test)}
""")

Model Performance (Accuracy):
Train: 1.0
Test: 0.85



# Conclusion

The results of this experiment cannot be reliably interpreted due to insufficient sample sizes in both training and testing datasets. The limited number of samples compromises the statistical validity of comparisons between traditional and contextual word embedding approaches, making it difficult to draw meaningful conclusions about their relative performance.

**Methodological Differences Between Embedding Types**
A fundamental distinction exists between the two embedding approaches tested:

Traditional word embeddings (such as Word2Vec, GloVe, or FastText) require

1.   training from scratch on domain-specific corpora to capture relevant semantic relationships
2.   Contextual embeddings (such as BERT, ELMo, or other transformer-based models) can be directly applied as pre-trained models without additional training, leveraging their existing learned representations


This difference in training requirements creates an inherent asymmetry in the experimental setup that may have influenced the comparative results.

**Cross-lingual Considerations**

While the majority of pre-training data for popular language models has historically been in English, modern embedding architectures increasingly incorporate multilingual training data, including Bahasa Indonesia. This multilingual training enables better performance on non-English tasks, but the proportion and quality of Bahasa Indonesia data in the pre-training corpus can significantly impact model performance.

**Recommendations for Future Research**
To conduct a more robust comparison:


1.   Increase sample sizes to meet statistical significance requirements
2.   Ensure fair comparison by either training traditional embeddings on adequate data or using pre-trained traditional embeddings when available
3.   Consider language-specific models trained specifically on Bahasa Indonesia corpora
4.   Document training data composition to understand potential biases toward specific languages
5.   Implement cross-validation to ensure results are generalizable


The experiment highlights the importance of careful experimental design when comparing fundamentally different embedding approaches, particularly in multilingual contexts.